# Neuron SWC quality control with `standard_morph`

`standard_morph` runs QC on SWC reconstructions in two phases:

- **Input integrity** — is the raw SWC *table* even well-formed enough to build a morphology? (required columns, unique node ids, valid parent references, acyclic, ...). These read the raw table.
- **Morphology quality** — given a buildable table, does the *neuron* look right? (single root, connectivity, edge lengths, compartment typing, tortuosity, position in the CCF, ...). These read a built graph.

One call does both: `run_qc(swc, context, suite_name=...)` resolves a suite, runs each phase, and returns a structured `RunReport`. The design goal is that a malformed file yields a normal report naming the problem instead of a stack trace. Some metrics will still run even if there are failed integrity tests — for instance, if node ids are duplicated, coordinate-only checks like `nodes_outside_ccf_mesh` still run while topology-dependent ones are skipped.

## Three things define a run: the *suite*, the *context*, and the *policy*

- **A suite** (or an explicit `metrics=[...]` list) — *which* checks to run.
- **A `QCContext`** — *the world those checks run in* (coordinate space, morphology kind, resources, ...).
- **A policy** — *how strict* the checks are (numeric thresholds OR upper/lower bounding threshold). Selected by `context.policy_version` (default `"policy_v1"`), overridable per call with `run_qc(..., policy_version=...)`.

### The context

| field | what it declares | example |
|---|---|---|
| `space` | coordinate space the cell lives in | `Space.IMAGE_SPACE`, `Space.CCF_REGISTERED` |
| `morphology_kind` | what the SWC represents | `MorphologyKind.AXON` / `DENDRITE` / `MERGED` |
| `resources` | external inputs a check needs but the SWC doesn't contain | `{}`, `{"ccf_atlas_path": ...}` |
| `policy_version` | which threshold set to apply | `"policy_v1"` |

Before running anything, `run_qc` checks every requested metric against the context and **fails fast** with `IncompatibleMetricContextError` if any is incompatible (wrong space, missing resource) rather than silently skipping it or crashing partway through a batch. Requesting `nodes_outside_ccf_mesh` in `image_space` raises the error up front.

### Resources

`resources` is a dict of inputs a check needs that don't live in the SWC file — an atlas volume, a soma MIP image, the source filename. A metric that needs one lists the key in `required_resources`; if that key is absent, the run halts fail-fast. Examples:

- `resources={"ccf_atlas_path": "/path/to/annotation_25.nrrd"}` — custom atlas for CCF-mesh checks (set `ccf_resolution` to match).
- `resources={"ccf_annotation": my_ndarray}` — inject an already-loaded annotation volume so a batch doesn't re-read a multi-GB file on every call.
- `resources={"image_soma_xyz": ..., "image_soma_radius_xyz": ...}` — soma centroid + radius for `soma_at_centroid`.

`run_qc` auto-injects `resources["filename"]` when you pass a file path, so filename-format checks work without extra wiring.

### The policy

A **policy** is a versioned bundle of numeric thresholds — the "how strict" knob, held separately from the measurement code. Thresholds can be space-keyed: `edge_length` is `{"image_space": 30.0, "ccf_registered": 10.0}` so the same metric applies the right bar for the context space. Add a new `policy_vN` rather than editing an existing one to keep old reports reproducible.


In [1]:
import json
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from standard_morph import (
    run_qc, QCContext, Space, MorphologyKind, REGISTRY,
    read_swc, PreparedMorphology,
    available_suites, available_policies, get_policy,
    load_ccf_annotation, clear_atlas_cache,
)
from standard_morph.suites import resolve_suite
from standard_morph.engine import BUILDABILITY_METRICS
from standard_morph.exceptions import IncompatibleMetricContextError, MissingPolicyValueError, MissingPolicyValuesError

## Library discovery

Before running anything, explore what's available: suites, registered metrics, and policy thresholds.

In [2]:
print("Suites:", available_suites())
print()
for s in available_suites():
    names = resolve_suite(s)
    print(f"{s} ({len(names)} metrics):")
    for n in names:
        print(f"  {n}")
    print()

print("All registered metrics:")
print(REGISTRY.names())

Suites: ['default_post_registration_tests', 'default_pre_registration_tests']

default_post_registration_tests (14 metrics):
  single_root_node
  soma_first_node
  single_connected_component
  node_identity_types
  duplicate_node_coordinates
  branch_max_degree
  edge_length
  soma_child_distance
  axon_origination
  apical_origination
  compartment_transitions
  local_tortuosity
  soma_inside_ccf_mesh
  nodes_outside_ccf_mesh

default_pre_registration_tests (12 metrics):
  single_root_node
  soma_first_node
  single_connected_component
  node_identity_types
  duplicate_node_coordinates
  branch_max_degree
  edge_length
  soma_child_distance
  axon_origination
  apical_origination
  compartment_transitions
  local_tortuosity

All registered metrics:
['acyclic', 'apical_origination', 'axon_origination', 'branch_max_degree', 'castable_columns', 'compartment_transitions', 'duplicate_node_coordinates', 'edge_length', 'filename_format', 'local_tortuosity', 'node_identity_types', 'nodes_outs

### Policy: inspect thresholds

In [10]:
print("Policies:", available_policies())
policy = get_policy("policy_v1")

for k,v in policy.thresholds.items():
    print(f"{k}: {v}")
    

Policies: ['policy_v1']
local_tortuosity: {'tortuosity_threshold': 10.0}
branch_max_degree: {'max_children': 2}
edge_length: {'max_length_um': {'image_space': PolicyRange(lo=0, hi=30.0), 'ccf_registered': PolicyRange(lo=0, hi=10.0)}}
soma_child_distance: {'max_soma_child_to_soma_um': 50.0}
axon_origination: {'max_axon_origin_to_soma_um': 75.0}
apical_origination: {'max_origins': 1}
compartment_transitions: {}
filename_format: {'name_format': 'AIND'}
soma_at_centroid: {'max_offset_fraction': 0.5}
single_connected_component: {}
single_root_node: {}
soma_first_node: {}
valid_parent_references: {}
acyclic: {}
parent_before_child: {}
node_identity_types: {'allowed_types': [1, 2, 3, 4]}
duplicate_node_coordinates: {}
nodes_outside_ccf_mesh: {'max_fraction_outside': 0.05}
soma_inside_ccf_mesh: {}


## What gets checked

The engine always runs the integrity checks first, then the morphology checks defined by the suite. Here is the full pre-registration lineup, tagged by phase.

In [11]:
suite = "default_pre_registration_tests"

def describe(names):
    rows = [{"metric": n,
             "phase": REGISTRY.get(n).evaluation_phase.value,
             "description": REGISTRY.get(n).display_name}
            for n in names]
    return pd.DataFrame(rows)

describe(BUILDABILITY_METRICS + resolve_suite(suite))

,metric,phase,description
0,required_columns,input_integrity,Required SWC columns present
1,non_empty,input_integrity,SWC table is non-empty
2,castable_columns,input_integrity,Column values cast to required types
3,unique_node_ids,input_integrity,Node ids are unique
4,valid_parent_references,input_integrity,Parent references exist
5,acyclic,input_integrity,Parent chain is acyclic
6,single_root_node,morphology_quality,Single root node validation
7,soma_first_node,morphology_quality,Soma is the first node
8,single_connected_component,morphology_quality,Single connected component
9,node_identity_types,morphology_quality,Expected node identity types


## QC a single reconstruction


In [15]:
raw_data_dir = Path("../wnm_ingestion_runner/ingested_data/RawDataUm")

example = sorted(raw_data_dir.glob("*.swc"))[0]

context = QCContext(
    space=Space.IMAGE_SPACE,
    morphology_kind=MorphologyKind.DENDRITE,
    resources={},
    policy_version='policy_v1'
)

suite = "default_pre_registration_tests"
report = run_qc(str(example), context, suite_name=suite)

print(example.name)
print(f"integrity ok: {report.integrity_ok}  |  overall: {report.summary['overall_status']}  |  passed: {report.passed}")
report.summary

18455_108839-X9999-Y9999_um.swc
integrity ok: True  |  overall: pass  |  passed: True


{'n_metrics': 12,
 'n_pass': 12,
 'n_fail': 0,
 'n_review': 0,
 'n_error': 0,
 'n_skipped': 0,
 'overall_status': 'pass'}

Can send the report object to a python dictionary and visualize/write to json

In [16]:
for t in report.to_dict()['integrity_results']:
    print(json.dumps(t, indent=2))
    print()

{
  "name": "required_columns",
  "status": "pass",
  "message": "All required SWC columns are present.",
  "value": 0,
  "value_label": "n_missing_columns",
  "thresholds_used": {},
  "measurements": {
    "required_columns": [
      "node_id",
      "compartment",
      "x",
      "y",
      "z",
      "r",
      "parent"
    ],
    "present_columns": [
      "node_id",
      "compartment",
      "x",
      "y",
      "z",
      "r",
      "parent"
    ],
    "missing_columns": []
  },
  "flagged_node_ids": [],
  "flagged_node_coordinates": [],
  "counts": {
    "n_missing_columns": 0
  },
  "artifacts": [],
  "runtime_ms": 0.010299998393747956
}

{
  "name": "non_empty",
  "status": "pass",
  "message": "SWC table has 59 node(s).",
  "value": 59,
  "value_label": "n_rows",
  "thresholds_used": {},
  "measurements": {
    "n_rows": 59
  },
  "flagged_node_ids": [],
  "flagged_node_coordinates": [],
  "counts": {
    "n_rows": 59
  },
  "artifacts": [],
  "runtime_ms": 0.0023000029614

In [17]:
for t in report.to_dict()['results']:
    print(json.dumps(t, indent=2))
    print('----------------------')

{
  "name": "single_root_node",
  "status": "pass",
  "message": "Single valid root: one soma-type root, node_id 1.",
  "value": 1,
  "value_label": "n_roots",
  "thresholds_used": {},
  "measurements": {
    "n_soma_roots": 1,
    "n_roots": 1,
    "n_type1_nodes": 1
  },
  "flagged_node_ids": [],
  "flagged_node_coordinates": [],
  "counts": {
    "n_roots": 1,
    "n_type1_nodes": 1,
    "n_flagged": 0
  },
  "artifacts": [],
  "runtime_ms": 0.016600002709310502
}
----------------------
{
  "name": "soma_first_node",
  "status": "pass",
  "message": "First node (id 1) is the soma.",
  "value": true,
  "value_label": "soma_is_first_node",
  "thresholds_used": {},
  "measurements": {
    "first_node_id": 1,
    "first_node_compartment": 1,
    "soma_is_first_node": true
  },
  "flagged_node_ids": [],
  "flagged_node_coordinates": [],
  "counts": {},
  "artifacts": [],
  "runtime_ms": 0.0023000029614195228
}
----------------------
{
  "name": "single_connected_component",
  "status": "

Each metric returns a `MetricResult`: a status, an optional headline `value` (a max, count, or fraction meant for trending), and — when it fails — the exact node ids it flagged. Flattening both phases into one table is the most useful per-cell view.

In [18]:
def results_frame(report):
    rows = [{"metric": r.name,
             "status": r.status,
             "value": r.value,
             "value_label": r.value_label,
             "n_flagged": len(r.flagged_node_ids),
             "flagged_node_ids": r.flagged_node_ids,
             "flagged_node_coordinates": r.flagged_node_coordinates,
             "message": r.message}
            for r in report.integrity_results + report.results]
    return pd.DataFrame(rows)

results_frame(report)

,metric,status,value,value_label,n_flagged,flagged_node_ids,flagged_node_coordinates,message
0,required_columns,pass,0,n_missing_columns,0,[],[],All required SWC columns are present.
1,non_empty,pass,59,n_rows,0,[],[],SWC table has 59 node(s).
2,castable_columns,pass,0,n_uncastable_values,0,[],[],All column values cast cleanly to their requir...
3,unique_node_ids,pass,0,n_duplicate_ids,0,[],[],All node ids are unique.
4,valid_parent_references,pass,0,n_dangling_parents,0,[],[],All parent ids reference an existing node.
5,acyclic,pass,0,n_cyclic_nodes,0,[],[],Parent chain is acyclic.
6,single_root_node,pass,1,n_roots,0,[],[],"Single valid root: one soma-type root, node_id 1."
7,soma_first_node,pass,True,soma_is_first_node,0,[],[],First node (id 1) is the soma.
8,single_connected_component,pass,1,n_components,0,[],[],Reconstruction is a single connected component.
9,node_identity_types,pass,0,n_unexpected_type_nodes,0,[],[],"Node types [1, 3] are all expected."


## Inspecting morphology structure with `PreparedMorphology`

`PreparedMorphology` is the array-backed structure every morphology-quality metric runs on. Build one directly for quick topological inspection — roots, tips, branch points, segments, connected components — without running a full QC suite.

In [19]:
df = read_swc(str(example))
pm = PreparedMorphology.from_dataframe(df)

print(f"nodes:           {pm.n}")
print(f"roots:           {pm.roots.size}  (indices into pm: {pm.roots.tolist()})")
print(f"tips:            {pm.tips.size}")
print(f"branch points:   {pm.branch_points.size}")
print(f"connected comps: {pm.n_components}")
print(f"segments:        {len(pm.segments)}")

# Map internal indices back to original SWC node_ids
branch_ids = [int(pm.node_id[i]) for i in pm.branch_points[:5]]
print(f"\nfirst branch node_ids: {branch_ids}")

nodes:           59
roots:           1  (indices into pm: [0])
tips:            3
branch points:   2
connected comps: 1
segments:        5

first branch node_ids: [3, 9]


## Topology-scope failure: selective skipping

A duplicate node id breaks the id→index map, making tree structure untrustworthy. `unique_node_ids` fails with scope `TOPOLOGY` — only metrics that need the tree are skipped; coordinate/attribute-only ones still run and report.

In [20]:
# Inject a duplicate node id to trigger a topology-scope failure.
# unique_node_ids fails → topology-dependent metrics become "skipped".
# Coordinate-only metrics (nodes_outside_ccf_mesh, duplicate_node_coordinates) still run.
bad_df = read_swc(str(example)).copy()
bad_df.loc[bad_df.index[-1], "node_id"] = bad_df.loc[bad_df.index[0], "node_id"]

dup_report = run_qc(bad_df, context, suite_name=suite)
print(f"integrity_ok: {dup_report.integrity_ok}")
print()
results_frame(dup_report)[["metric", "status", "message"]]

integrity_ok: False



,metric,status,message
0,required_columns,pass,All required SWC columns are present.
1,non_empty,pass,SWC table has 59 node(s).
2,castable_columns,pass,All column values cast cleanly to their requir...
3,unique_node_ids,fail,duplicate node_id(s): [1]
4,valid_parent_references,pass,All parent ids reference an existing node.
5,acyclic,fail,parent chain contains a cycle involving 19 nod...
6,single_root_node,skipped,not run: topology is unreliable (e.g. duplicat...
7,soma_first_node,pass,First node (id 1) is the soma.
8,single_connected_component,skipped,not run: topology is unreliable (e.g. duplicat...
9,node_identity_types,pass,"Node types [1, 3] are all expected."


## Opt-in checks: `filename_format` and `parent_before_child`

These are not in any default suite — request them explicitly via `metrics=[...]`. Both are input-integrity checks that are **non-blocking**: they fail the report but never skip downstream morphology checks.

In [21]:
def find_result(report, name):
    for r in report.integrity_results + report.results:
        if r.name == name:
            return r

# Filename convention (opt-in, not in any default suite).
# When passing a path, resources["filename"] is auto-injected.
fname_report = run_qc(str(example), QCContext(space=Space.IMAGE_SPACE), metrics=["filename_format"])
r = find_result(fname_report, "filename_format")
print(f"filename_format ({example.name}):")
print(f"  status:  {r.status}")
print(f"  message: {r.message}")

# parent_before_child — report only, never blocks downstream checks.
# Supply the filename explicitly when passing a DataFrame.
df = read_swc(str(example))
pb_report = run_qc(
    df,
    QCContext(space=Space.IMAGE_SPACE, resources={"filename": example.name}),
    metrics=["parent_before_child"],
)
r2 = find_result(pb_report, "parent_before_child")
print(f"\nparent_before_child:")
print(f"  status:  {r2.status}")
print(f"  message: {r2.message}")

filename_format (18455_108839-X9999-Y9999_um.swc):
  status:  fail
  message: Filename '18455_108839-X9999-Y9999_um.swc' does not match the AIND naming convention.

parent_before_child:
  status:  pass
  message: Every parent appears before its children.


## QC a whole directory

Write one JSON report per cell (the full record, good for auditing or a database) plus a flat summary CSV (one row per file, one column per check) for at-a-glance triage. The `try/except` is belt-and-suspenders: `run_qc` already turns bad *data* into a report, so a raise here means something genuinely unexpected (an unreadable file, a misconfig) — we log it and keep going rather than losing the batch.

In [22]:
import os
out_dir = raw_data_dir.parent / "RawDataUm_QCSummary"
out_dir.mkdir(parents=True, exist_ok=True)

rows = []
reports = {}
for swc in tqdm(sorted(raw_data_dir.glob("*.swc"))):
    try:
        report = run_qc(str(swc), context, suite_name=suite)
    except Exception as exc:
        rows.append({"file": swc.name, "overall_status": "ERROR",
                     "message": f"{type(exc).__name__}: {exc}"})
        continue
    reports[os.path.basename(str(swc))] = report
    (out_dir / f"{swc.stem}.qc.json").write_text(json.dumps(report.to_dict(), indent=2))

    row = {"file": swc.name,
           "overall_status": report.summary["overall_status"],
           "integrity_ok": report.integrity_ok,
           "passed": report.passed}
    for r in report.integrity_results + report.results:
        row[r.name] = r.status
    rows.append(row)

summary = pd.DataFrame(rows)
summary.to_csv(out_dir / "qc_summary.csv", index=False)
print(f"{len(summary)} files -> {out_dir}")
summary.head()

100%|██████████| 304/304 [00:29<00:00, 10.37it/s]

304 files -> ..\wnm_ingestion_runner\ingested_data\RawDataUm_QCSummary


,file,overall_status,integrity_ok,passed,required_columns,non_empty,castable_columns,unique_node_ids,valid_parent_references,acyclic,...,single_connected_component,node_identity_types,duplicate_node_coordinates,branch_max_degree,edge_length,soma_child_distance,axon_origination,apical_origination,compartment_transitions,local_tortuosity
0,18455_108839-X9999-Y9999_um.swc,pass,True,True,pass,pass,pass,pass,pass,pass,...,pass,pass,pass,pass,pass,pass,pass,pass,pass,pass
1,18455_109928-X9999-Y9999_um.swc,pass,True,True,pass,pass,pass,pass,pass,pass,...,pass,pass,pass,pass,pass,pass,pass,pass,pass,pass
2,18455_110722-X9999-Y9999_um.swc,pass,True,True,pass,pass,pass,pass,pass,pass,...,pass,pass,pass,pass,pass,pass,pass,pass,pass,pass
3,18455_110888-X9999-Y9999_um.swc,fail,True,False,pass,pass,pass,pass,pass,pass,...,pass,pass,fail,pass,pass,pass,pass,pass,pass,pass
4,18455_111050-X9999-Y9999_um.swc,pass,True,True,pass,pass,pass,pass,pass,pass,...,pass,pass,pass,pass,pass,pass,pass,pass,pass,pass


## Where do cells fail?

`overall_status` is `pass` / `fail` / `incomplete` (a blocking integrity check fired) / `error`.

In [24]:
summary["overall_status"].value_counts()

overall_status
fail          243
pass           52
incomplete      9
Name: count, dtype: int64

In [25]:
_incomplete_df = summary[summary['overall_status'] == "incomplete"].head()
_cols = [c for c in _incomplete_df.columns if not all([v in ['pass','fail'] for v in _incomplete_df[c]])]
_incomplete_df[_cols]

,file,overall_status,integrity_ok,passed,branch_max_degree
46,18455_153044-X9999-Y9999_um.swc,incomplete,True,False,review
49,18455_21284-X9999-Y9999_um.swc,incomplete,True,False,review
61,191812_42817-X9999-Y9999_um.swc,incomplete,True,False,review
63,191816_10930-X9999-Y9999_um.swc,incomplete,True,False,review
98,192334_14803-X9999-Y9999_um.swc,incomplete,True,False,review


In [26]:
fail_example = _incomplete_df.file.values[0]
report_results = reports[fail_example].to_dict()['results']
branch_max_degree_result = [r for r in report_results if r['name'] == 'branch_max_degree'][0]
branch_max_degree_result

{'name': 'branch_max_degree',
 'status': 'review',
 'message': '5 branch point(s) have more than 2 children.',
 'value': 3,
 'value_label': 'max_children_observed',
 'thresholds_used': {'max_children': 2},
 'measurements': {'max_children_observed': 3},
 'flagged_node_ids': [3, 156, 205, 229, 258],
 'flagged_node_coordinates': [[5399.0, 4603.0, 4318.0],
  [5244.0, 4542.0, 3949.0],
  [5432.0, 4476.0, 3824.0],
  [5379.0, 4544.0, 3786.0],
  [5398.0, 4599.0, 4300.0]],
 'counts': {'n_flagged': 5},
 'artifacts': [],
 'runtime_ms': 0.02510000194888562}

In [27]:
fail_example_df = pd.read_csv(os.path.join(raw_data_dir, fail_example),
            sep='\s',
            index_col=0,
            header=None, 
            names=['node_id','type','x','y','z','radius','parent_id'])

for node_id in branch_max_degree_result['flagged_node_ids']:
    node_row = fail_example_df.loc[fail_example_df.index == node_id]
    num_children = fail_example_df[fail_example_df['parent_id']==node_id]
    print(f"node_id: {node_id}  |  num_children: {len(num_children)}  |  type: {node_row.type.values[0]}")

node_id: 3  |  num_children: 3  |  type: 3
node_id: 156  |  num_children: 3  |  type: 2
node_id: 205  |  num_children: 3  |  type: 2
node_id: 229  |  num_children: 3  |  type: 2
node_id: 258  |  num_children: 3  |  type: 2


C:\Users\matt.mallory\AppData\Local\Temp\ipykernel_31352\4028730394.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  fail_example_df = pd.read_csv(os.path.join(raw_data_dir, fail_example),


## Post-registration checks

Once a reconstruction is registered to the CCF, the post-registration suite adds two atlas-aware checks:
- `soma_inside_ccf_mesh` — soma voxel falls inside the CCF brain mesh
- `nodes_outside_ccf_mesh` — fraction of all nodes outside the brain (`max_fraction_outside` threshold)

**CCF-registered SWC files must have coordinates in micron space** — the standard output of all CCF registration pipelines. The library converts micron coordinates to atlas voxels internally using `floor(coord / resolution)`.

The bundled 10 µm Allen CCF atlas is used by default — no extra config is needed. `ccf_resolution` only needs to be set when supplying a custom atlas at a different voxel size:

```python
# Default: bundled 10 µm atlas
QCContext(space=Space.CCF_REGISTERED)

# Custom atlas at a different resolution
QCContext(space=Space.CCF_REGISTERED, ccf_resolution=25,
          resources={"ccf_atlas_path": "/path/to/annotation_25.nrrd"})
```

The space-agnostic checks (edge length, tortuosity, etc.) run in CCF space too — same suite, same call, just tighter thresholds where the policy says so.

In [30]:
registered_dir = Path("../wnm_ingestion_runner/ingested_data/RegisteredData/202602161429_upload_resampled_reconstructions_192334_1")

# CCF-registered SWCs must be in micron space.
# ccf_resolution defaults to 10 (the bundled Allen CCF atlas) — no extra config needed.
ccf_context = QCContext(
    space=Space.CCF_REGISTERED,
    morphology_kind=MorphologyKind.MERGED,
    resources={},
    policy_version='policy_v1'
)

reg_example = sorted(registered_dir.glob("*.swc"))[0]
reg_report = run_qc(str(reg_example), ccf_context, suite_name="default_post_registration_tests")

print(reg_example.name, "->", reg_report.summary["overall_status"])
reg_report_dict = reg_report.to_dict()
print("Integrty results that did not pass:")
[c for c in reg_report_dict['integrity_results'] if c['status']!='pass']
print("Metric results that did not pass:")
for t in [c for c in reg_report_dict['results'] if c['status']!='pass']:
    print(json.dumps(t, indent=2))


192334_14803-X9999-Y9999_reg.swc -> fail
Integrty results that did not pass:
Metric results that did not pass:
{
  "name": "branch_max_degree",
  "status": "review",
  "message": "4 branch point(s) have more than 2 children.",
  "value": 4,
  "value_label": "max_children_observed",
  "thresholds_used": {
    "max_children": 2
  },
  "measurements": {
    "max_children_observed": 4
  },
  "flagged_node_ids": [
    380,
    385,
    404,
    433
  ],
  "flagged_node_coordinates": [
    [
      6519.826,
      3213.776,
      5446.752
    ],
    [
      6515.848,
      3226.854,
      5450.848
    ],
    [
      6516.192,
      3200.069,
      5439.575
    ],
    [
      6512.915,
      3225.402,
      5455.174
    ]
  ],
  "counts": {
    "n_flagged": 4
  },
  "artifacts": [],
  "runtime_ms": 0.022799998987466097
}
{
  "name": "edge_length",
  "status": "fail",
  "message": "184/484 edge(s) outside [0, 10.0] um (min 1.3, max 33.4 um).",
  "value": 0.38016528925619836,
  "value_label": "f

### Fail-fast: incompatible context

Requesting a CCF metric in `image_space` raises `IncompatibleMetricContextError` before any work is done — no partial runs, no silent skips.

In [31]:
# Fail-fast demo: requesting a CCF-only metric in image_space raises immediately.
try:
    run_qc(str(reg_example), QCContext(space=Space.IMAGE_SPACE),
           metrics=["nodes_outside_ccf_mesh"])
except IncompatibleMetricContextError as e:
    print("IncompatibleMetricContextError:", e)

IncompatibleMetricContextError: One or more requested metrics are incompatible with the run context:
  - Metric 'nodes_outside_ccf_mesh' is incompatible with context: space 'image_space' not in allowed ['ccf_registered']


## Batch CCF QC with atlas caching

The 10 µm annotation atlas decompresses to ~4.8 GB. It is loaded once and cached automatically, but for a large batch it's faster to warm it up front and inject the array directly so no file I/O happens per cell.

In [32]:
# The ~4.8 GB 10 µm atlas loads once per process and is cached.
# Warm it up front and inject the array so every cell in the batch reuses it.
annotation = load_ccf_annotation(resolution=10)
batch_ccf_context = QCContext(
    space=Space.CCF_REGISTERED,
    morphology_kind=MorphologyKind.MERGED,
    resources={"ccf_annotation": annotation},
)

rows = []
for swc in tqdm(sorted(registered_dir.glob("*.swc"))):
    try:
        rep = run_qc(str(swc), batch_ccf_context, suite_name="default_post_registration_tests")
    except Exception as exc:
        rows.append({"file": swc.name, "overall_status": "ERROR", "message": str(exc)})
        continue
    row = {"file": swc.name, "overall_status": rep.summary["overall_status"], "passed": rep.passed}
    for r in rep.results:
        row[r.name] = r.status
    rows.append(row)

clear_atlas_cache()  # free ~4.8 GB when done

ccf_summary = pd.DataFrame(rows)
print(f"{len(ccf_summary)} registered cells")
ccf_summary.head()

100%|██████████| 4/4 [00:00<00:00, 17.17it/s]

4 registered cells


,file,overall_status,passed,single_root_node,soma_first_node,single_connected_component,node_identity_types,duplicate_node_coordinates,branch_max_degree,edge_length,soma_child_distance,axon_origination,apical_origination,compartment_transitions,local_tortuosity,soma_inside_ccf_mesh,nodes_outside_ccf_mesh
0,192334_14803-X9999-Y9999_reg.swc,fail,False,pass,pass,pass,pass,pass,review,fail,pass,pass,pass,pass,pass,pass,pass
1,192334_14836-X9999-Y9999_reg.swc,fail,False,pass,pass,pass,pass,fail,review,fail,pass,pass,pass,pass,pass,pass,pass
2,192334_19917-X9999-Y9999_reg.swc,fail,False,pass,pass,pass,pass,fail,review,fail,pass,pass,pass,pass,pass,pass,pass
3,192334_6730-X9999-Y9999_reg.swc,fail,False,pass,pass,pass,pass,pass,pass,fail,pass,pass,pass,pass,pass,pass,pass


Break it down per check — which metrics account for most of the failures?

In [33]:
metric_cols = [c for c in summary.columns
               if c not in {"file", "overall_status", "integrity_ok", "passed", "message"}]

status_by_metric = (summary[metric_cols]
                    .apply(pd.Series.value_counts)
                    .T.fillna(0).astype(int))
order = [c for c in ["pass", "fail", "skipped", "error"] if c in status_by_metric.columns]
sort_col = "fail" if "fail" in order else order[0]
status_by_metric[order].sort_values(sort_col, ascending=False)

,pass,fail
duplicate_node_coordinates,82,222
edge_length,133,171
local_tortuosity,275,29
required_columns,304,0
non_empty,304,0
castable_columns,304,0
single_root_node,304,0
unique_node_ids,304,0
valid_parent_references,304,0
acyclic,304,0


Then drill into one problem cell straight from its saved JSON — status, message, and the first few flagged nodes for anything that didn't pass.

In [34]:
failed = summary[summary["overall_status"].isin(["fail", "incomplete", "error"])]
print(len(failed), "files with something to look at")

if len(failed):
    name = failed.iloc[0]["file"]
    saved = json.loads((out_dir / f"{Path(name).stem}.qc.json").read_text())
    print("\n" + name)
    for r in saved["integrity_results"] + saved["results"]:
        if r["status"] != "pass":
            print(f"  {r['status']:8} {r['name']:26} {r['message']}")
            flagged = r["flagged_node_ids"][:10]
            if flagged:
                print(f"           flagged nodes (first 10): {flagged}")

252 files with something to look at

18455_110888-X9999-Y9999_um.swc
  fail     duplicate_node_coordinates 2 node(s) in 1 group(s) share coordinates with another node.
           flagged nodes (first 10): [18, 19]
